# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Oguzhandyr/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Grain (Unit of Analysis):** Exactly one row = one daily performance record for a pseudonymized URL (`content_hash_id`) belonging to a client (`client_hash_id`) on a specific date (`report_date`).
* **Time Window:** Mid-panel observation month `2026-03` (2026-03-01 to 2026-03-31) used for feature building and validation. The final month `2026-06` is treated as the sealed holdout test set.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Features (Knowable pre-decision):**
  * `gsc_impressions`: Historical search impressions in Google Search Console.
  * `gsc_clicks`: Historical organic search clicks.
  * `gsc_avg_position`: Mean organic ranking position.
  * `ctr`: Historic click-through rate (`gsc_clicks / nullif(gsc_impressions, 0)`).
  * `sessions_organic`: GA4 organic landing page visits.
* **Label (Target / Proxy):**
  * `is_declining_proxy`: Binary decay indicator (`1` if `gsc_avg_position > 20`, `0` otherwise).
* **Context (Identifiers):**
  * `client_hash_id`, `content_hash_id`, `report_date`.
* **Excluded (Target Leakage / Post-decision):**
  * `trend_pct` / future performance deltas (computed from future outcome windows).
  * Algorithmic recommendation flags (`needs_ctr_fix`, `health_score`).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score


try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.getenv("HF_TOKEN")

con = duckdb.connect()
if hf_token:
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
slice_path = f"{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet"

print("--- Step 1: Three Verification Queries on month=2026-03 ---")


df_grain = con.sql(f"""
    SELECT
        count(*) as total_rows,
        count(distinct client_hash_id) as unique_clients,
        count(distinct content_hash_id) as unique_content_items
    FROM read_parquet('{slice_path}')
""").df()
print("Query 1 (Grain):")
print(df_grain)


df_span = con.sql(f"""
    SELECT
        count(*) as total_rows,
        min(report_date) as start_date,
        max(report_date) as end_date
    FROM read_parquet('{slice_path}')
""").df()
print("\nQuery 2 (Row Count & Date Span):")
print(df_span)


df_avail = con.sql(f"""
    SELECT
        count(*) as total_rows,
        count(case when (gsc_impressions > 0) IS TRUE then 1 end) as active_impressions,
        count(case when (gsc_avg_position IS NOT NULL) IS TRUE then 1 end) as valid_positions
    FROM read_parquet('{slice_path}')
""").df()
print("\nQuery 3 (Availability with IS TRUE):")
print(df_avail)

print("\n--- Step 2: Honest 5 Features vs. Deliberate Leakage Trap ---")


local_df = con.sql(f"""
    SELECT
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        (gsc_clicks * 1.0 / nullif(gsc_impressions, 0)) as ctr,
        sessions_organic,
        case when gsc_avg_position > 20 then 1 else 0 end as is_declining_proxy,
        gsc_avg_position as leaky_exact_target
    FROM read_parquet('{slice_path}')
    WHERE gsc_impressions > 0
    LIMIT 20000
""").df().fillna(0)

features_honest = ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ctr", "sessions_organic"]
X_honest = local_df[features_honest]
y = local_df["is_declining_proxy"]


clf_honest = DecisionTreeClassifier(max_depth=2, random_state=42)
clf_honest.fit(X_honest, y)
pred_honest = clf_honest.predict(X_honest)
print(f"Honest Model Precision: {precision_score(y, pred_honest, zero_division=0):.4f}")


features_leaky = features_honest + ["leaky_exact_target"]
X_leaky = local_df[features_leaky]
clf_leaky = DecisionTreeClassifier(max_depth=2, random_state=42)
clf_leaky.fit(X_leaky, y)
pred_leaky = clf_leaky.predict(X_leaky)
print(f"Trap (Leaky) Model Precision: {precision_score(y, pred_leaky, zero_division=0):.4f} <-- Artificial 1.0 from leakage!")

del X_leaky
print("Trap removed. Honest feature contract preserved.")

--- Step 1: Three Verification Queries on month=2026-03 ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 (Grain):
   total_rows  unique_clients  unique_content_items
0     9841378              55                331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query 2 (Row Count & Date Span):
   total_rows start_date   end_date
0     9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query 3 (Availability with IS TRUE):
   total_rows  active_impressions  valid_positions
0     9841378             3611061          3611061

--- Step 2: Honest 5 Features vs. Deliberate Leakage Trap ---
Honest Model Precision: 1.0000
Trap (Leaky) Model Precision: 1.0000 <-- Artificial 1.0 from leakage!
Trap removed. Honest feature contract preserved.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Tracking Discrepancies:** GA4 `sessions_organic` is zero for client profiles lacking analytics access.
* **Aggregated Window Blur:** Daily performance aggregates do not reflect intra-day query position shifts.
* **Lack of Off-Page Signals:** Backlinks, competitor indexation updates, and algorithm release timelines are unobserved in this warehouse slice.

In [6]:
print("--- Data Limits & Signal Coverage ---")
df_limits = con.sql(f"""
    SELECT
        count(*) as total_rows,
        count(case when sessions_organic = 0 or sessions_organic is null then 1 end) as missing_ga4_sessions,
        count(case when gsc_clicks = 0 then 1 end) as zero_click_rows
    FROM read_parquet('{slice_path}')
""").df()
print(df_limits)

--- Data Limits & Signal Coverage ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  missing_ga4_sessions  zero_click_rows
0     9841378               9628735          9423397


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.